# Proof of Concept -- Analyst & Portfolio Manager


## Two step financial advisor and assistant 

In [ ]:
import os

#!pip install openai
from openai import OpenAI

In [ ]:
# Environment setup, replace with your own API key
os.environ["OPENAI_API_KEY"] = "GET_YOUR_OWN_KEY"

client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),
    #base_url=API_BASE_URL
)
CHAT_MODEL = "gpt-4.1-mini"
EMBED_MODEL = "text-embedding-3-small"

In [3]:
def llm(prompt: str) -> str:
   
    from openai import OpenAI
    client = OpenAI()

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a financial analyst."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content

In [4]:
# 3. Tiny mock "news corpus" for the RAG step
# In the real project I’ll swap this with a proper vector store over real news.

corpus = [
    {
        "id": "doc1",
        "title": "Company X Q3 earnings preview",
        "content": """
        Analysts expect Company X to report modest revenue growth but 
        pressure on margins due to higher input costs. Key catalysts include 
        a new product line in Q4 and potential regulatory approval in Europe.
        """,
        "source": "Mock Financial Times",
    },
    {
        "id": "doc2",
        "title": "Macro headwinds for consumer discretionary",
        "content": """
        Rising interest rates and slowing consumer demand may weigh on 
        discretionary spending. Companies with high leverage and weak pricing 
        power are particularly at risk.
        """,
        "source": "Mock Bloomberg",
    },
    {
        "id": "doc3",
        "title": "Social media buzz around Company X",
        "content": """
        Retail investors on social media discuss Company X as a potential 
        short squeeze candidate. Sentiment is mixed but volatility expectations are high.
        """,
        "source": "Mock Twitter",
    },
]


Mock Rag step

In [5]:
def retrieve_documents(query: str, top_k: int = 3) -> List[Dict]:
    query_terms = set(query.lower().split())
    scored = []
    for doc in corpus:
        text = (doc["title"] + " " + doc["content"]).lower()
        score = sum(1 for t in query_terms if t in text)
        scored.append((score, doc))
    # sort by descending score and take top_k
    scored = sorted(scored, key=lambda x: x[0], reverse=True)
    return [doc for score, doc in scored[:top_k] if score > 0] or [d for _, d in scored[:top_k]]

## Analyst Step

In [12]:
@dataclass
class AnalystMemo:
    ticker: str
    thesis: str
    catalysts: List[str]
    risks: List[str]
    valuation_context: str
    time_horizon: str
    overall_sentiment: str
    sources_used: List[str]

def format_docs_for_prompt(docs: List[Dict]) -> str:
    blocks = []
    for d in docs:
        blocks.append(
            f"Title: {d['title']}\nSource: {d['source']}\nContent:\n{textwrap.dedent(d['content']).strip()}\n"
        )
    return "\n\n---\n\n".join(blocks)

In [13]:

def run_analyst_agent(opportunity: str, ticker: str = "COMPX") -> AnalystMemo:
    docs = retrieve_documents(opportunity, top_k=3)
    docs_str = format_docs_for_prompt(docs)
    
    prompt = f"""
    You are a fundamental equity analyst.

    The investment opportunity is:
    '{opportunity}'

    You are given a set of news-like documents about this company and macro context:

    {docs_str}

    Produce a concise, structured analysis with the following fields:

    - Investment thesis (2–4 sentences)
    - Key catalysts (bullet points)
    - Key risks (bullet points)
    - Valuation context (1–3 sentences; qualitative is fine)
    - Time horizon (e.g. '3–6 months', '1–3 years')
    - Overall sentiment (one of: 'bullish', 'cautiously bullish', 'neutral', 'bearish')

    Respond strictly in this JSON format (no extra text):

    {{
      "ticker": "{ticker}",
      "thesis": "...",
      "catalysts": ["...", "..."],
      "risks": ["...", "..."],
      "valuation_context": "...",
      "time_horizon": "...",
      "overall_sentiment": "...",
      "sources_used": ["title or id of documents you used"]
    }}
    """

    raw = llm(prompt)

    import json
    data = json.loads(raw)
    memo = AnalystMemo(**data)
    return memo

In [8]:
# 7. Mock portfolio & mandate

mock_portfolio = [
    {"ticker": "COMPX", "weight": 0.0,  "sector": "Consumer Discretionary", "max_weight": 0.05},
    {"ticker": "MEGACORP", "weight": 0.08, "sector": "Tech", "max_weight": 0.10},
    {"ticker": "DEFSAFE",  "weight": 0.04, "sector": "Utilities", "max_weight": 0.06},
]

portfolio_mandate = {
    "max_single_name_weight": 0.05,
    "max_sector_weight": 0.25,
    "target_volatility": "medium",
    "drawdown_limit": "12%",
    "style": "long-only, fundamentally driven, 1–3 year horizon",
}


In [ ]:
@dataclass
class PMDecision:
    decision: str # "Add", "Reduce", "Hold", "Reject"
    size_change: float # e.g. +0.02 means +2% notional weight
    confidence: float # between 0 and 1
    rationale: str # explanation 
    expected_impact: str # impact on risk, concentration, drawdown

def format_portfolio_for_prompt(portfolio: List[Dict]) -> str:
    lines = []
    for p in portfolio:
        lines.append(
            f"{p['ticker']}: weight={p['weight']:.2%}, "
            f"sector={p['sector']}, max_weight={p['max_weight']:.2%}"
        )
    return "\n".join(lines)

def format_mandate_for_prompt(mandate: Dict) -> str:
    return "\n".join(f"{k}: {v}" for k, v in mandate.items())


In [ ]:
def run_pm_agent(memo: AnalystMemo) -> PMDecision:
    portfolio_str = format_portfolio_for_prompt(mock_portfolio)
    mandate_str = format_mandate_for_prompt(portfolio_mandate)
    
    prompt = f"""
    You are a portfolio manager for a long-only fundamental equity fund.

    Here is the current portfolio (weights are portfolio weights):
    {portfolio_str}

    Mandate and constraints:
    {mandate_str}

    Analyst memo for {memo.ticker}:
    Thesis: {memo.thesis}
    Catalysts: {memo.catalysts}
    Risks: {memo.risks}
    Valuation context: {memo.valuation_context}
    Time horizon: {memo.time_horizon}
    Overall sentiment: {memo.overall_sentiment}

    Task:
    - Decide whether to: "Add", "Reduce", "Hold", or "Reject" the position in {memo.ticker}.
      (If weight is currently 0, 'Add' or 'Reject' makes most sense.)
    - Propose a size change in portfolio weight (e.g. +0.02 for +2%, -0.01 for -1%).
    - Provide a confidence score between 0 and 1.
    - Explain your reasoning in 3–6 sentences, explicitly linking back to the memo AND the constraints.
    - Describe qualitatively the expected impact on portfolio risk, concentration, and drawdown.

    Respond strictly in this JSON format:

    {{
      "decision": "Add | Reduce | Hold | Reject",
      "size_change": 0.02,
      "confidence": 0.7,
      "rationale": "...",
      "expected_impact": "..."
    }}
    """

    raw = llm(prompt)
    
    import json
    data = json.loads(raw)
    decision = PMDecision(**data)
    return decision


In [11]:
investment_opportunity = "Company X is releasing earnings soon."

analyst_memo = run_analyst_agent(investment_opportunity, ticker="COMPX")
pm_decision = run_pm_agent(analyst_memo)

print("=== ANALYST MEMO ===")
for k, v in asdict(analyst_memo).items():
    print(f"{k}: {v}\n")

print("\n=== PM DECISION ===")
for k, v in asdict(pm_decision).items():
    if isinstance(v, float) and k in {"size_change", "confidence"}:
        print(f"{k}: {v:.2f}")
    else:
        print(f"{k}: {v}")

=== ANALYST MEMO ===
ticker: COMPX

thesis: Company X is poised for modest revenue growth in the upcoming earnings report, driven by a new product line and potential regulatory approval. However, margin pressures from rising input costs and macroeconomic headwinds may limit upside potential.

catalysts: ['New product line launching in Q4', 'Potential regulatory approval in Europe']

risks: ['Higher input costs impacting margins', 'Rising interest rates and slowing consumer demand affecting discretionary spending']

valuation_context: The current valuation may reflect cautious sentiment due to macroeconomic challenges, but upcoming catalysts could provide a re-rating if executed successfully.

time_horizon: 3–6 months

overall_sentiment: cautiously bullish

sources_used: ['Company X Q3 earnings preview', 'Social media buzz around Company X', 'Macro headwinds for consumer discretionary']


=== PM DECISION ===
decision: Add
size_change: 0.02
confidence: 0.70
rationale: Given the cautiousl